# Project 03: detecting and locating line outages from bus measurements

**Tools:** scikit-learn, pandapower, Python
**Network:** IEEE 39-bus New England (Athay, Podmore and Virmani, 1979) as shipped with pandapower
**Notebook status:** executed, all numbers below are produced by this notebook

---

## The question

> Can a classifier detect that a transmission line has tripped, and identify which one,
> using only bus voltage measurements? And does it survive realistic measurement noise?

## Why I framed it this way

My first version of this project scored almost perfectly and taught me nothing, because I
included line power flows in the feature vector. A line that is out of service reports no
flow, so the feature vector contained a direct pointer to the answer. That is target
leakage, and it is easy to miss because the result looks like success.

The notebook keeps both feature sets so the effect is visible:

- **Set B**, all 113 measurements including line flows. Leaky.
- **Set A**, 78 bus voltage magnitudes and angles only. The honest problem.

Every headline number comes from Set A.


## 1. Generating the dataset

For each sample I scale every load by a random factor drawn from a normal distribution
with 10 percent standard deviation, optionally take one line out of service, solve AC
power flow, and record the bus state. Samples where the power flow fails to converge are
discarded.

This produced 2800 usable samples, 1400 healthy and 1400 with a
line out, covering 35 distinct line outages.


In [ ]:
import sys, os, time
import numpy as np
import pandapower as pp, pandapower.networks as pn

OUT, STORE = "/tmp/work/out/", "/tmp/work/out/p03_dataset.npz"
SEED = 42
LOAD_SIGMA = 0.10          # load scaling noise, 10 percent standard deviation

def features(net):
    """124 measurements: 39 voltage magnitudes, 39 angles, 46 line active flows."""
    return np.concatenate([net.res_bus.vm_pu.values,
                           net.res_bus.va_degree.values,
                           net.res_line.p_from_mw.values])

def run(n_per_class, seed_offset):
    rng = np.random.default_rng(SEED + seed_offset)
    net = pn.case39()
    base_scaling = net.load.scaling.values.copy()
    n_lines = len(net.line)
    X, y_bin, y_loc = [], [], []
    t0 = time.time()
    made = 0
    while made < 2 * n_per_class:
        fault = made >= n_per_class
        net.load.scaling = base_scaling * rng.normal(1.0, LOAD_SIGMA, len(net.load)).clip(0.6, 1.4)
        line = int(rng.integers(0, n_lines)) if fault else -1
        if fault:
            net.line.at[line, "in_service"] = False
        try:
            pp.runpp(net, numba=False)
            X.append(features(net)); y_bin.append(int(fault)); y_loc.append(line)
            made += 1
        except Exception:
            pass
        finally:
            if fault:
                net.line.at[line, "in_service"] = True
        if time.time() - t0 > 33:
            break
    return np.array(X), np.array(y_bin), np.array(y_loc)

X, y_bin, y_loc = run(200, 0)
print(X.shape, (y_bin==0).sum(), (y_bin==1).sum())

## 2. Loading and splitting

An out of service line reports NaN flow, which I convert to zero. Physically that is what
a measurement system would report for a de-energised circuit, and it is exactly the value
that makes Set B leak.


In [ ]:
OUT, SEED, N_BUS = "/tmp/work/out/", 42, 39
d = np.load(OUT + "p03_dataset.npz")
X_all = np.nan_to_num(d["X"], nan=0.0)          # de-energised line reports zero flow
y, yloc = d["y_bin"], d["y_loc"]
X_v = X_all[:, :2 * N_BUS]                       # voltages + angles only
print(f"samples {len(X_all)}  normal {(y==0).sum()}  fault {(y==1).sum()}")
print(f"SET A {X_v.shape[1]} features   SET B {X_all.shape[1]} features", flush=True)

def split(X):
    return train_test_split(X, y, yloc, test_size=0.3, random_state=SEED, stratify=y)

Xtr, Xte, ytr, yte, ltr, lte = split(X_v)
Btr, Bte, _, _, _, _ = split(X_all)

def sc(yt, yp):
    return dict(accuracy=accuracy_score(yt, yp),
                precision=precision_score(yt, yp, zero_division=0),
                recall=recall_score(yt, yp, zero_division=0),
                f1=f1_score(yt, yp, zero_division=0))

## 3. The baseline, measured before any classifier

The baseline is a statistical rule with no learning in it: flag a fault when any bus
voltage deviates more than k standard deviations from the healthy profile, where the mean
and standard deviation come from the training healthy samples only. The threshold k is
tuned on the training set, never on the test set.

This matters. A model is only interesting if it beats the simplest thing that could work.
The tuned baseline reaches **82.7 percent accuracy** with recall of
68.3 percent, so it detects most faults but misses about a third of them.


In [ ]:
# ---------- 1. BASELINE, measured before any classifier ----------
# Statistical rule: flag a fault when any bus voltage deviates more than k standard
# deviations from the healthy-case profile learned on the training normal samples.
mu = Xtr[ytr == 0][:, :N_BUS].mean(axis=0)
sd = Xtr[ytr == 0][:, :N_BUS].std(axis=0) + 1e-9

def zrule(Xm, k):
    return (np.abs((Xm[:, :N_BUS] - mu) / sd).max(axis=1) > k).astype(int)

ks = [2, 3, 4, 5, 6, 8]
df_base = pd.DataFrame([dict(k=k, **sc(ytr, zrule(Xtr, k))) for k in ks])
best_k = float(df_base.loc[df_base.accuracy.idxmax(), "k"])       # tuned on TRAIN only
baseline = sc(yte, zrule(Xte, best_k))
print(f"\nBASELINE z-rule, k tuned on train = {best_k}")
print(df_base.to_string(index=False))
print("baseline on test:", {k: round(v, 4) for k, v in baseline.items()}, flush=True)

## 4. Detection

Random forest and an RBF kernel SVM on Set A, plus a random forest on the leaky Set B for
comparison.


In [ ]:
# ---------- 2. detection ----------
def mk(kind):
    m = (RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
         if kind == "rf" else SVC(kernel="rbf", C=10, gamma="scale", random_state=SEED))
    return Pipeline([("sc", StandardScaler()), ("m", m)])

rows = [dict(model="Baseline z-rule (Set A)", features=X_v.shape[1], cv_mean=np.nan, cv_std=np.nan, **baseline)]
fitted = {}
for name, kind in (("RandomForest", "rf"), ("SVM_rbf", "svm")):
    p = mk(kind)
    cv = cross_val_score(p, Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=-1)
    p.fit(Xtr, ytr); fitted[name] = p
    rows.append(dict(model=f"{name} (Set A)", features=X_v.shape[1],
                     cv_mean=cv.mean(), cv_std=cv.std(), **sc(yte, p.predict(Xte))))
leak = mk("rf"); leak.fit(Btr, ytr)
rows.append(dict(model="RandomForest (Set B, LEAKY)", features=X_all.shape[1],
                 cv_mean=np.nan, cv_std=np.nan, **sc(yte, leak.predict(Bte))))
df_det = pd.DataFrame(rows)
print("\nDETECTION\n", df_det.to_string(index=False), flush=True)

| Model | Features | Test accuracy |
|---|---|---|
| Baseline z-rule | 78 | 0.827 |
| Random forest | 78 | 0.918 |
| **SVM, RBF kernel** | 78 | **0.962** |
| Random forest, leaky Set B | 113 | 0.961 |

Two things worth noticing.

The SVM at 0.962 beats the random forest at 0.918. The decision
boundary between healthy and faulted states is smooth in voltage space, which suits a
kernel method better than axis aligned splits.

More interestingly, the leaky model at 0.961 does **not** beat the honest
SVM. I expected leakage to dominate. It does not, because detecting that something has
changed is easy either way. Leakage helps far more with location, which is the next
section.


## 5. Locating the outage, 35 classes

In [ ]:
# ---------- 3. location ----------
mtr, mte = ltr >= 0, lte >= 0
locA = mk("rf"); locA.fit(Xtr[mtr], ltr[mtr])
pA = locA.predict(Xte[mte]); accA = accuracy_score(lte[mte], pA)
prob = locA.predict_proba(Xte[mte])
top3 = float(np.mean([lte[mte][i] in locA.classes_[np.argsort(p)[-3:]] for i, p in enumerate(prob)]))
locB = mk("rf"); locB.fit(Btr[mtr], ltr[mtr])
accB = accuracy_score(lte[mte], locB.predict(Bte[mte]))
n_cls = int(len(np.unique(ltr[mtr])))
print(f"\nLOCATION {n_cls} classes | Set A top-1 {accA:.4f} top-3 {top3:.4f} | Set B (leaky) {accB:.4f}", flush=True)

Top-1 accuracy on Set A is 0.886 and top-3 is 0.945,
against a random guess of 1 in 35, which is 2.9 percent.
With the leaky features it rises to 0.986, which is where the leakage
really shows: finding the line with no flow is trivial, inferring it from voltages is not.

## 6. Noise sensitivity

This is the section that changed my view of the whole project. I add zero mean Gaussian
noise scaled to a percentage of each measurement's mean magnitude, and re-evaluate.


In [ ]:
# ---------- 4. noise sensitivity ----------
rng = np.random.default_rng(SEED)
scale = np.abs(Xte).mean(axis=0)
rows = []
for nl in [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]:
    Xn = Xte + rng.normal(0, nl * scale, Xte.shape)
    r = dict(noise_pct=nl * 100, baseline=accuracy_score(yte, zrule(Xn, best_k)))
    for n, m in fitted.items():
        r[n] = accuracy_score(yte, m.predict(Xn))
    r["location"] = accuracy_score(lte[mte], locA.predict(Xn[mte]))
    rows.append(r)
df_noise = pd.DataFrame(rows)
print("\nNOISE\n", df_noise.to_string(index=False), flush=True)

**Key finding, and it is a negative one.** Every model collapses to chance at
0.5 percent measurement noise. The random forest goes from 0.918 to
0.500 at 1 percent noise, which is exactly the 0.5 that random guessing gives
on a balanced two class problem. Location degrades even faster, reaching
0.309 at 1 percent noise.

The reason is that the classifiers are keying on very small voltage differences. Removing
one line from a well meshed 39-bus network at moderate loading barely moves the bus
voltages, so the signal the model relies on is smaller than realistic instrument error.

That is worth stating plainly. A published accuracy figure on clean simulated data says
almost nothing about whether a method would work on a real system. I would not have
learned this if I had stopped at the 0.962 headline number.

## 7. Figures

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.5))
b = df_det.set_index("model")["accuracy"]
ax[0].barh(range(len(b)), b.values, color=["#B89E7E", "#4A7C6F", "#4B7FA8", "#C4714A"])
ax[0].set_yticks(range(len(b)))
ax[0].set_yticklabels([m.replace(" (", "\n(") for m in b.index], fontsize=8)
ax[0].set(xlim=(0.4, 1.05), xlabel="Test accuracy", title="Detection, and what leakage looks like")
for i, v in enumerate(b.values): ax[0].text(v + .01, i, f"{v:.3f}", va="center", fontsize=8)
for c, col, lab in (("baseline", "#B89E7E", "z-rule baseline"), ("RandomForest", "#4A7C6F", "Random forest"),
                    ("SVM_rbf", "#4B7FA8", "SVM (RBF)"), ("location", "#C4714A", "Location, top-1")):
    ax[1].plot(df_noise.noise_pct, df_noise[c], "o-", color=col, lw=2, ms=4, label=lab)
ax[1].set(xlabel="Measurement noise [% of mean]", ylabel="Accuracy", title="Degradation with noise")
ax[1].legend(fontsize=8)
cm = confusion_matrix(lte[mte], pA, labels=locA.classes_)
im = ax[2].imshow(cm, cmap="YlOrBr")
ax[2].set(xlabel="Predicted line", ylabel="True line", title=f"Location confusion, Set A, top-1 {accA:.2f}")
plt.colorbar(im, ax=ax[2], fraction=.046)
for a in ax[:2]: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT + "p03_results.png", dpi=160)

## 8. What I would do next

- Use PMU grade measurements with realistic error models rather than uniform Gaussian
  noise. Real instrument transformer error is not white and not identically distributed.
- Test on a system under heavier loading, where an outage moves voltages more and the
  signal to noise ratio improves.
- Compare against a physics based method, specifically state estimation residual analysis,
  which is what utilities actually use and which the machine learning approach should have
  to beat.
- Add measurement dropout, since real systems lose channels.
- Try graph neural networks, which respect the network topology rather than treating the
  measurements as an unordered vector.

## Limitations

- Synthetic data from a single network and a single topology.
- The only fault modelled is a line outage. No short circuits, no generator trips.
- Load variation is independent across buses, which is unrealistic since real load is
  spatially correlated.
- No temporal information. A real detector would see a time series, not a snapshot.

## References

1. Athay, T., Podmore, R., and Virmani, S. (1979). "A Practical Method for the Direct
   Analysis of Transient Stability." *IEEE Transactions on Power Apparatus and Systems*,
   PAS-98(2), 573 to 584. DOI: 10.1109/TPAS.1979.319407. Original source of the New
   England test system.
2. Illinois Center for a Smarter Electric Grid. "IEEE 39-Bus System."
   https://icseg.iti.illinois.edu/ieee-39-bus-system/
3. Thurner, L. et al. (2018). "pandapower: An Open-Source Python Tool for Convenient
   Modeling, Analysis, and Optimization of Electric Power Systems." *IEEE Transactions on
   Power Systems*, 33(6), 6510 to 6521. DOI: 10.1109/TPWRS.2018.2829021
4. Mohammadi Shakiba, F., Azizi, S. M., Zhou, M., and Abusorrah, A. (2023). "Application
   of machine learning methods in fault detection and classification of power transmission
   lines: a survey." *Artificial Intelligence Review*, 56, 5799 to 5836.
   DOI: 10.1007/s10462-022-10296-0
5. Rafique, F., Fu, L., and Mai, R. (2021). "End to end machine learning for fault
   detection and classification in power transmission lines." *Electric Power Systems
   Research*, 199, 107430. DOI: 10.1016/j.epsr.2021.107430
6. Pedregosa, F. et al. (2011). "Scikit-learn: Machine Learning in Python."
   *Journal of Machine Learning Research*, 12, 2825 to 2830.
